# Working Memory for Spatial Location

**Author:** Teo Fantacci  
**Purpose:** Pedagogical companion to Froudist-Walsh et al. (in press, *Biol Psychiatry*) — illustrates the firing-rate **bump-attractor network** for spatial working memory following Wimmer et al. (2014). Corresponds to **Fig 3E** and **§3.4** of the review.

> **This notebook is a Python port of the MATLAB code released by Wimmer et al. (2014).**  
> Specifically, it is a faithful, line-by-line translation of *Supplementary Code 1* (`41593_2014_BFnn3645_MOESM23_ESM.txt`). Parameter values, equations, integration scheme, and noise conventions are taken **verbatim** from the released `.m` file — see the "What this notebook is, and is not" section for the four conventions inherited from the MATLAB source that differ from the rest of this tutorial series.

## How to cite
If you find this code useful for your research, please cite:
- The original paper (and its MATLAB code): Wimmer K, Nykamp DQ, Constantinidis C, Compte A (2014): Bump attractor dynamics in prefrontal cortex explains behavioral precision in spatial working memory. *Nat Neurosci* 17: 431–439.
- The review: Froudist-Walsh S, Ivanov TG, Magrou L, Cho H, Busch AN, Muller LE, et al. (in press): Mechanistic computational models of cognition across scales and species. *Biol Psychiatry*.

## Requirements
```bash
pip install numpy matplotlib
```
Runs in Google Colab (recommended) or locally with Jupyter (Python ≥ 3.8).  
No external data files required — all simulations are self-contained.

## Notes
All code is a Python re-implementation of the published MATLAB source. No third-party data or figures are included. Simulation outputs are generated at runtime and not stored in the repository.


## Model overview

This notebook implements the **firing-rate bump-attractor network** described in Wimmer K, Nykamp DQ, Constantinidis C, Compte A (2014): Bump attractor dynamics in prefrontal cortex explains behavioral precision in spatial working memory. *Nat Neurosci* 17: 431–439. Specifically, we faithfully reproduce **Supplementary Code 1** (`41593_2014_BFnn3645_MOESM23_ESM.txt`), the MATLAB source code released with the paper.

The model is the firing-rate variant used in **Figure 8** of the paper to test, against alternative hypotheses (discrete attractor, decaying bump), whether prefrontal persistent activity during the delay of an oculomotor delayed-response task can be explained by a continuous bump attractor on a ring of neurons. The same architecture had previously been introduced as a *spiking* network by Compte, Brunel, Goldman-Rakic & Wang (2000).

## What this notebook is, and is not

**It is** a literal port of the MATLAB code, with no extra ingredients added. The four important things to know:

1. The transfer function is **not** the Wong–Wang Abbott–Chance form used in the other notebooks of this series. It is a **piecewise dimensionless** function from Brunel (2003): $f(x)=x^2$ for $0<x<1$, $f(x)=\sqrt{4x-3}$ for $x\geq 1$, $f(x)=0$ otherwise. Because $f$ is dimensionless, currents and rates in this model are **also dimensionless** — they are not in nA and Hz.
2. There are **no AMPA / NMDA / GABA gating variables**. Recurrent input is computed instantaneously from the current rates: the system is a pure Wilson–Cowan rate model with one time constant for E ($\tau_E=20$ ms) and one for I ($\tau_I=10$ ms).
3. **Inhibition is all‐to‐all uniform**: the recurrent inhibitory input is just $G_{IE}\langle r_I\rangle$ (and analogously $G_{II}\langle r_I\rangle$ for I-to-I). Only the E‐to‐E connectivity is structured (von Mises on the ring).
4. **Noise is additive Gaussian white noise on the rate equation**, not OU-filtered noise on the input current.

We reproduce all four choices verbatim. A subsequent notebook (in preparation) will replace these ingredients with the Wong–Wang machinery (Abbott–Chance transfer function, AMPA/NMDA/GABA gating, OU noise on currents) used in the rest of this tutorial series.

## Task simulated

The protocol mimics the oculomotor delayed-response task of Funahashi, Bruce and Goldman‐Rakic (1989). One trial consists of:

1. **Pre-stimulus / fixation** (1 s): only background input + noise.
2. **Cue** (500 ms): a localised input is delivered to E neurons whose preferred location is near the cue angle.
3. **Delay** (2 s): the network is left autonomous. A bump of selective activity is expected to persist around the cue location, although it diffuses slowly because of noise.
4. **Reset** (500 ms): a global negative input is applied to all E neurons to wipe persistent activity.

Total: 4.2 s. (Note: the paper Methods describe a 1 + 0.5 + 3 s trial; the released MATLAB code uses 1 + 0.5 + 2 s + reset. We follow the code.)

The behavioural read-out is the angle of the **population vector** computed from the E rates.

## Setup

Standard imports and Drive / local fallback for saving figures, matching the style of the other notebooks in this series.

In [ ]:
import os

# If running in Google Colab, optionally point this to your own Drive folder.
# Otherwise leave as './' — the notebook will work in the current directory.
tutorial_path = './'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Optional: uncomment and edit the line below to save to your Drive instead
    # tutorial_path = '/content/drive/MyDrive/your_folder_here/'
    print(f"Running in Colab. Working in: {tutorial_path}")
except (ImportError, ModuleNotFoundError):
    tutorial_path = os.getcwd()
    print(f"Running locally. Working in: {tutorial_path}")

SAVE_DIR = os.path.join(tutorial_path, 'saved_results')
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Results will be saved under: {SAVE_DIR}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

rng = np.random.default_rng(seed=42)

## Parameters

Every value below is **copied verbatim** from `41593_2014_BFnn3645_MOESM23_ESM.txt` (Wimmer 2014 Supplementary Code 1). We keep the exact same names and units (including the ones that are dimensionless).

Lighter than the paper for speed: the .m script visualises every step in real time. Here we just store traces and plot afterwards. The numerical scheme and parameter values are identical.

In [ ]:
PARAMS = {
    # Network size
    'N': 512,                  # number of "neurons" per population (E and I)

    # Timing (ms)
    'totalTime': 4200.0,       # total simulation duration
    'dt': 2.0,                 # integration step
    'stimon': 1000.0,          # cue onset
    'stimoff': 1500.0,         # cue offset
    'delayend': 3500.0,        # end of delay period (start of reset)

    # Membrane / rate time constants (ms)
    'tauE': 20.0,              # excitatory rate time constant
    'tauI': 10.0,              # inhibitory rate time constant

    # Connectivity
    'kappa': 1.5,              # concentration of the von Mises E-to-E footprint
    'GEE': 6.0,                # excitation to excitatory neurons (structured, scaled by W_E)
    'GEI': 4.0,                # excitation to inhibitory neurons (uniform: uses mean(rE))
    'GIE': 3.4,                # inhibition to excitatory neurons (uniform: uses mean(rI))
    'GII': 0.85,               # inhibition to inhibitory neurons (uniform: uses mean(rI))

    # Background drive
    'I0E': 0.2,                # constant external input to E
    'I0I': 0.5,                # constant external input to I

    # Noise (additive Gaussian on the rate equation, not OU)
    'sigE': 1.0,               # std-dev of noise added to E rates each step
    'sigI': 3.0,               # std-dev of noise added to I rates each step

    # Stimulus
    'stim': 200.0,             # peak amplitude of the cue (also used for the global reset)
    'cue_angle': 0.0,          # cue location (rad, on the ring [-pi, pi]); demo uses 0
}

for k, v in PARAMS.items():
    print(f'  {k:12s} = {v}')

## Building blocks

Three small helpers that mirror the Matlab code one-for-one.

### 1. Connectivity matrix

The E‐to‐E weight from neuron $j$ to neuron $i$ depends only on their angular distance $\Delta\theta_{ij}$. The Matlab code uses a **von Mises kernel** $v(\theta) \propto \exp(\kappa\cos\theta)$ normalised to sum to 1 over all neurons, and then realises $W^{EE}$ as a **circulant matrix** built from $v$. This is mathematically equivalent to $W^{EE}_{ij}=v(\theta_i-\theta_j)$.

### 2. Transfer function

The piecewise function from Brunel (2003), used as-is by Wimmer:

$$f(x) = \begin{cases} 0 & \text{if } x \le 0 \\ x^2 & \text{if } 0 < x < 1 \\ \sqrt{4x-3} & \text{if } x \ge 1 \end{cases}$$

Continuous and continuously differentiable at $x=1$ (both branches give $f(1)=1$ and $f'(1)=2$), with a smoother saturation than a pure square.

### 3. Population vector decoder

Standard read-out: the angle of $\sum_i r^E_i\,e^{i\theta_i}$. This is what the model treats as the ‘behavioural response’.

In [ ]:
def make_connectivity(N: int, kappa: float) -> np.ndarray:
    """Build the von-Mises E-to-E connectivity matrix on a ring of N neurons.

    Returns W with rows indexed by postsynaptic neuron, columns by presynaptic.
    Each row sums to 1 (normalised von Mises footprint), exactly as in the .m file.
    """
    theta = np.arange(N) / N * 2 * np.pi          # [0, 2*pi)
    v = np.exp(kappa * np.cos(theta))
    v = v / v.sum()                                # normalise to sum 1
    # circulant: row i = circular shift of v by i positions
    # Matlab gallery('circul', v) places v in row 0, then each next row
    # is v shifted by one index. We mirror that convention.
    idx = (np.arange(N)[:, None] - np.arange(N)[None, :]) % N
    W = v[idx]
    return W


def f_transfer(x: np.ndarray) -> np.ndarray:
    """Piecewise dimensionless transfer function from Brunel (2003)."""
    out = np.zeros_like(x, dtype=float)
    mid = (x > 0) & (x < 1)
    high = x >= 1
    out[mid] = x[mid] ** 2
    out[high] = np.sqrt(4.0 * x[high] - 3.0)
    return out


def decode_angle(r: np.ndarray, theta: np.ndarray) -> float:
    """Population-vector angle of rates r over preferred angles theta."""
    return float(np.arctan2(np.sum(r * np.sin(theta)), np.sum(r * np.cos(theta))))

Quick visual check of the connectivity profile and the transfer function. These are the two most important model ingredients to internalise.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

# Connectivity row plotted vs. true post-pre angular distance.
# Row 0 of W is W[0, j] = v[(0 - j) mod N], so column j corresponds to a
# pre-synaptic neuron at angle theta_j = j * 360 / N (relative to neuron 0).
# We wrap theta_j to (-180, 180] so the peak (j = 0) sits at 0 deg, with no
# 180-deg offset ambiguity.
N = PARAMS['N']
W = make_connectivity(N, PARAMS['kappa'])
dtheta = np.arange(N) / N * 360.0
dtheta = ((dtheta + 180.0) % 360.0) - 180.0          # wrap to (-180, 180]
order = np.argsort(dtheta)
ax = axes[0]
ax.plot(dtheta[order], W[0][order], color='C3')
ax.set_xlabel(r'$\theta_{post}-\theta_{pre}$ (deg)')
ax.set_ylabel(r'$W^{EE}_{ij}$ (normalised)')
ax.set_title(rf'E-to-E footprint, $\kappa$={PARAMS["kappa"]}')
ax.axvline(0, ls=':', color='k', alpha=0.3)
ax.set_xlim(-180, 180)

# Transfer function
x = np.linspace(-0.5, 4.0, 400)
ax = axes[1]
ax.plot(x, f_transfer(x), color='C0')
ax.axvline(1.0, ls=':', color='k', alpha=0.3)
ax.set_xlabel('input $x$ (dimensionless)')
ax.set_ylabel('rate $f(x)$ (dimensionless)')
ax.set_title(r'Transfer function: $x^2$ for $x<1$, $\sqrt{4x-3}$ for $x\geq 1$')
ax.set_xlim(-0.5, 4.0)

plt.tight_layout()
plt.show()

## Single-trial simulator

The integration is the **forward Euler** step used in the .m file:

$$r^E_i(t+\Delta t) = r^E_i(t) + \frac{\Delta t}{\tau_E}\big[\,f(I^E_i(t)) - r^E_i(t) + \sigma_E\,\xi^E_i(t)\,\big]$$

with input

$$I^E_i(t) = G_{EE}\sum_j W^{EE}_{ij}\,r^E_j + I_{0,E} - G_{IE}\,\langle r^I\rangle + I^{ext}_i(t)$$

and analogously for I (with $W$ replaced by an all-to-all uniform coupling, expressed via $\langle r^E\rangle$).

The noise $\xi$ is *not* multiplied by $\sqrt{\Delta t}$ in the .m file. We keep that exact convention so the noise level is identical to the published simulation; this is a deliberate departure from the more standard Euler–Maruyama scaling used in the Wong–Wang notebooks.

We expose the relevant signals so we can plot rich diagnostics afterwards: the full E and I spatiotemporal activity, the decoded angle through time, and the components of the input current to E.

In [ ]:
def run_trial(params: dict, cue_angle: float | None = None,
              seed: int | None = None,
              record_currents: bool = False) -> dict:
    """Simulate one trial of the Wimmer 2014 bump-attractor network.

    Parameters
    ----------
    params : dict
        PARAMS dictionary.
    cue_angle : float or None
        Angle (rad in [-pi, pi]) where the cue is centred. If None, uses
        params['cue_angle'].
    seed : int or None
        RNG seed. If None, uses an unseeded default RNG draw.
    record_currents : bool
        If True, additionally return the time courses of the three current
        components on E (recurrent excitation, recurrent inhibition, external)
        for one neuron near the cue. Useful for didactic plots.

    Returns
    -------
    dict with keys: 'rE' (T, N), 'rI' (T, N), 'time' (T,), 'theta' (N,),
        'angle' (T,), and optionally current components.
    """
    p = params
    N = p['N']
    dt = p['dt']
    nsteps = int(np.floor(p['totalTime'] / dt))

    # Time-step indices for events (matching the .m floor(_/dt) convention)
    stimon_step = int(np.floor(p['stimon'] / dt))
    stimoff_step = int(np.floor(p['stimoff'] / dt))
    delayend_step = int(np.floor(p['delayend'] / dt))
    cue_dur_steps = stimoff_step - stimon_step

    # Connectivity
    W = make_connectivity(N, p['kappa'])

    # Preferred angles on the ring [-pi, pi)
    theta = np.arange(N) / N * 2 * np.pi - np.pi

    # Cue spatial profile: same von Mises shape as connectivity, peak amplitude `stim`
    if cue_angle is None:
        cue_angle = p.get('cue_angle', 0.0)
    v_cue = np.exp(p['kappa'] * np.cos(theta - cue_angle))
    v_cue = v_cue / v_cue.sum()
    stimulus = p['stim'] * v_cue

    # State variables
    rE = np.zeros(N)
    rI = np.zeros(N)

    # Storage
    rE_hist = np.zeros((nsteps, N))
    rI_hist = np.zeros((nsteps, N))
    angle_hist = np.zeros(nsteps)

    if record_currents:
        # Pick a probe neuron near the cue
        probe_idx = int(np.argmin(np.abs(theta - cue_angle)))
        Iexc_hist = np.zeros(nsteps)   # G_EE * (W rE)_probe
        Iinh_hist = np.zeros(nsteps)   # -G_IE * mean(rI)
        Iext_hist = np.zeros(nsteps)   # I0E + stimulus(probe) + reset

    rng_local = np.random.default_rng(seed)

    for i in range(nsteps):
        # Noise (one independent Gaussian per neuron per step, NOT scaled by sqrt(dt))
        noiseE = p['sigE'] * rng_local.standard_normal(N)
        noiseI = p['sigI'] * rng_local.standard_normal(N)

        # Recurrent input
        IE = p['GEE'] * (W @ rE) + (p['I0E'] - p['GIE'] * np.mean(rI))
        II = (p['GEI'] * np.mean(rE) - p['GII'] * np.mean(rI) + p['I0I']) * np.ones(N)

        # External, task-dependent input
        if stimon_step < i < stimoff_step:
            IE = IE + stimulus
        if delayend_step < i < delayend_step + cue_dur_steps:
            IE = IE - p['stim']           # global negative input to wipe the bump

        if record_currents:
            Iexc_hist[i] = p['GEE'] * (W[probe_idx] @ rE)
            Iinh_hist[i] = -p['GIE'] * np.mean(rI)
            Iext = p['I0E']
            if stimon_step < i < stimoff_step:
                Iext = Iext + stimulus[probe_idx]
            if delayend_step < i < delayend_step + cue_dur_steps:
                Iext = Iext - p['stim']
            Iext_hist[i] = Iext

        # Forward Euler step
        rE = rE + (f_transfer(IE) - rE + noiseE) * dt / p['tauE']
        rI = rI + (f_transfer(II) - rI + noiseI) * dt / p['tauI']

        rE_hist[i] = rE
        rI_hist[i] = rI
        angle_hist[i] = decode_angle(rE, theta)

    out = dict(
        rE=rE_hist, rI=rI_hist,
        time=np.arange(nsteps) * dt,
        theta=theta, angle=angle_hist,
        cue_angle=cue_angle,
        events=dict(stimon=p['stimon'], stimoff=p['stimoff'],
                    delayend=p['delayend'])
    )
    if record_currents:
        out.update(dict(Iexc=Iexc_hist, Iinh=Iinh_hist, Iext=Iext_hist,
                        probe_idx=probe_idx))
    return out

## Demo trial

Single trial with the cue at $0°$. We will look at the dynamics from several angles.

In [ ]:
trial = run_trial(PARAMS, cue_angle=0.0, seed=1, record_currents=True)
print(f'Simulation done. {len(trial["time"])} time steps, dt={PARAMS["dt"]} ms.')
print(f'Final E-rate range: [{trial["rE"][-1].min():.2f}, {trial["rE"][-1].max():.2f}]')
print(f'Final I-rate range: [{trial["rI"][-1].min():.2f}, {trial["rI"][-1].max():.2f}]')
print(f'Decoded angle right before reset: {np.degrees(trial["angle"][int(PARAMS["delayend"]/PARAMS["dt"])-1]):+.2f}°  (cue was 0°)')

### Plot 1 — Spatiotemporal activity of E and I populations

This is the canonical ‘raster-style’ view of the bump attractor: the y-axis is preferred angle, the x-axis is time, colour is firing rate. The vertical lines mark cue onset, cue offset, and end of delay (start of reset).

Three things to look for:
- **Before the cue**: roughly uniform low activity (the spontaneous state).
- **During cue + delay**: a localised ‘bump’ of E activity centred on $0°$.
- **After the reset**: bump abolished, network returns to spontaneous.

The I population is approximately uniform throughout, because inhibition in this model is all-to-all unstructured.

In [ ]:
def plot_spatiotemporal(trial: dict, save_name: str | None = None):
    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
    extent = [0, trial['time'][-1] / 1000.0, -180, 180]   # time in s

    for ax, data, label, cmap in zip(
        axes,
        [trial['rE'], trial['rI']],
        ['E population', 'I population'],
        ['hot', 'Blues'],
    ):
        im = ax.imshow(data.T, aspect='auto', origin='lower',
                       extent=extent, cmap=cmap,
                       vmin=0, vmax=np.percentile(data, 99.5))
        ax.set_ylabel(f'preferred angle (deg)\n{label}')
        for ev_name, ev_t in trial['events'].items():
            ax.axvline(ev_t / 1000.0, color='w', lw=0.8, ls='--')
        plt.colorbar(im, ax=ax, label='rate (a.u.)')
    axes[-1].set_xlabel('time (s)')
    axes[0].set_title(f'Spatiotemporal activity — cue at {np.degrees(trial["cue_angle"]):+.0f}°')
    plt.tight_layout()
    if save_name:
        for ext in ('.png', '.svg'):
            plt.savefig(os.path.join(SAVE_DIR, save_name + ext), dpi=150, bbox_inches='tight')
    plt.show()


plot_spatiotemporal(trial, save_name='wimmer_spatiotemporal_demo')

### Plot 2 — Snapshots of the E activity profile at key moments

Picking four moments tells the same story but in 1D, which sometimes makes the bump shape easier to read.

In [ ]:
def plot_snapshots(trial: dict, save_name: str | None = None):
    p_dt = PARAMS['dt']
    snapshot_times_ms = {
        'pre-cue (0.5 s)': 500.0,
        'late cue (1.4 s)': 1400.0,
        'mid delay (2.5 s)': 2500.0,
        'late delay (3.4 s)': 3400.0,
    }
    fig, ax = plt.subplots(figsize=(8.5, 4))
    theta_deg = np.degrees(trial['theta'])
    for label, t_ms in snapshot_times_ms.items():
        idx = int(t_ms / p_dt)
        ax.plot(theta_deg, trial['rE'][idx], label=label)
    ax.axvline(np.degrees(trial['cue_angle']), color='k', ls=':',
               alpha=0.4, label='cue location')
    ax.set_xlabel('preferred angle (deg)')
    ax.set_ylabel('E rate (a.u.)')
    ax.set_xlim(-180, 180)
    ax.legend(loc='upper right', fontsize=9)
    ax.set_title('E-population activity profile at four times')
    plt.tight_layout()
    if save_name:
        for ext in ('.png', '.svg'):
            plt.savefig(os.path.join(SAVE_DIR, save_name + ext), dpi=150, bbox_inches='tight')
    plt.show()


plot_snapshots(trial, save_name='wimmer_snapshots_demo')

### Plot 3 — Decoded angle through time

The population-vector decoder applied at every time step. During the pre-cue period it fluctuates randomly because the activity is essentially uniform; during cue and delay it locks onto the cue location and slowly drifts (the diffusion of the bump that Wimmer et al. claim explains behavioural inaccuracy).

In [ ]:
def plot_decoded_angle(trial: dict, save_name: str | None = None):
    fig, ax = plt.subplots(figsize=(10, 3.2))
    t_s = trial['time'] / 1000.0
    ax.plot(t_s, np.degrees(trial['angle']), color='C2', lw=0.8)
    ax.axhline(np.degrees(trial['cue_angle']), color='k', ls=':',
               alpha=0.5, label='cue location')
    for ev_name, ev_t in trial['events'].items():
        ax.axvline(ev_t / 1000.0, color='gray', lw=0.6, ls='--')
    ax.set_xlabel('time (s)')
    ax.set_ylabel('decoded angle (deg)')
    ax.set_ylim(-180, 180)
    ax.set_title('Population-vector read-out')
    ax.legend(loc='lower right')
    plt.tight_layout()
    if save_name:
        for ext in ('.png', '.svg'):
            plt.savefig(os.path.join(SAVE_DIR, save_name + ext), dpi=150, bbox_inches='tight')
    plt.show()


plot_decoded_angle(trial, save_name='wimmer_decoded_demo')

### Plot 4 — Synaptic input components for a probe E neuron near the cue

Decomposing the input the way Compte et al. (2000) Figure 3C does: recurrent excitation, recurrent inhibition, external input, and their sum. Outside the cue and reset windows, the *external* contribution is just the constant $I_{0,E}$. The interplay between recurrent excitation and recurrent inhibition is what supports persistent activity.

**A note on the reset pulse** (the deep negative excursion at $t\approx 3.5$ s in the bottom panel): the Matlab code applies a uniform $-\,\mathrm{stim} = -200$ to every E neuron for 500 ms after the delay ends. In the monkey experiment this corresponds to the **end of the delay period, when the fixation point disappears and the monkey makes the memory-guided saccade**. Compte et al. (2000) modelled this go-signal as a **depolarising** nonspecific input that excites both E and I neurons; because the network is inhibition-dominated, the consequent rise in inhibition is what wipes the bump. Wimmer's code takes a more direct shortcut: a **hyperpolarising** pulse applied straight to E. The net effect is the same — persistent activity is extinguished and the network returns to baseline, ready for the next trial — but Wimmer's version is a modelling convenience rather than a literal biological mechanism. The pulse is also large compared to the cue contribution on the gray curve because the cue is spread across the network through the normalised von Mises (so its peak on any single neuron is ${\sim}1$ a.u.) whereas the reset is applied uniformly at full $-200$ to every neuron.

The two panels below show the same trial: a zoom on the cue+delay window (top) where the E/I balance during persistent activity is visible, and the full trial (bottom) where the reset pulse dominates the y-axis.

In [ ]:
def plot_currents(trial: dict, save_name: str | None = None):
    """Two-panel view: the cue+delay window (top), and the full trial including reset (bottom).

    The reset pulse is ~200 a.u. in magnitude and would otherwise crush the
    y-axis of any single-axis plot, hiding the cue/delay balance.
    """
    if 'Iexc' not in trial:
        raise ValueError('Trial was not run with record_currents=True.')
    t_s = trial['time'] / 1000.0
    total = trial['Iexc'] + trial['Iinh'] + trial['Iext']
    probe_deg = np.degrees(trial['theta'][trial['probe_idx']])

    fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=False)
    cue_t = trial['events']['stimon'] / 1000.0
    delay_end_t = trial['events']['delayend'] / 1000.0

    for ax, (xlim, ylim, title) in zip(axes, [
        ((cue_t - 0.2, delay_end_t + 0.05), (-25, 40),
         f'Zoom on cue + delay (E neuron at {probe_deg:+.1f}°, cue at {np.degrees(trial["cue_angle"]):+.0f}°)'),
        ((0, t_s[-1]), None, 'Full trial (reset pulse visible at delay end)'),
    ]):
        ax.plot(t_s, trial['Iexc'], label='recurrent excitation', color='C3')
        ax.plot(t_s, trial['Iinh'], label='recurrent inhibition', color='C0')
        ax.plot(t_s, trial['Iext'], label='external (incl. cue, reset)', color='gray')
        ax.plot(t_s, total, label='total', color='k', lw=1.3)
        for ev_t in trial['events'].values():
            ax.axvline(ev_t / 1000.0, color='gray', lw=0.6, ls='--')
        ax.axhline(0, color='k', lw=0.4)
        ax.set_xlim(*xlim)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.set_ylabel('input (a.u.)')
        ax.set_title(title)
    axes[-1].set_xlabel('time (s)')
    axes[0].legend(loc='upper right', ncol=2, fontsize=9)
    plt.tight_layout()
    if save_name:
        for ext in ('.png', '.svg'):
            plt.savefig(os.path.join(SAVE_DIR, save_name + ext), dpi=150, bbox_inches='tight')
    plt.show()


plot_currents(trial, save_name='wimmer_currents_demo')

## Bump diffusion — small batch of trials

The central scientific claim of Wimmer et al. (2014) is that the *bump diffuses during the delay due to internal noise*, and that this diffusion is what produces variable (and inaccurate) memory-guided saccades. For a 1D diffusion process the **variance of the bump position should grow approximately linearly with time** within the delay (Compte 2000 Figure 5B, Wimmer 2014 Figure 5B).

Here we run a handful of trials with the same cue and overlay the decoded-angle trajectories. The expected pattern: all trajectories start at the cue at the end of the cue period, then spread apart over the delay, like trajectories of a 1D random walk.

**Sample-size caveat.** Sample variance is itself a random quantity: with $n=10$ trials the 95% interval on a variance estimate is roughly $[0.4\,\sigma^2,\,3.3\,\sigma^2]$ — almost an order of magnitude wide. So the variance curve from a small batch will fluctuate up and down rather than show a clean linear growth. **Run with `n_trials = 200` or more to see the linear trend cleanly.** We keep `n_trials = 10` here just for speed and to demonstrate the analysis pipeline.

In [ ]:
n_trials = 100
cue = 0.0
trials = [run_trial(PARAMS, cue_angle=cue, seed=100 + k) for k in range(n_trials)]
print(f'Ran {n_trials} trials.')

In [ ]:
def plot_drift(trials: list[dict], cue: float, save_name: str | None = None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6),
                              gridspec_kw=dict(width_ratios=[2, 1]))

    # Trajectories
    ax = axes[0]
    t_s = trials[0]['time'] / 1000.0
    delay_start = trials[0]['events']['stimoff'] / 1000.0
    delay_end = trials[0]['events']['delayend'] / 1000.0
    for tr in trials:
        ax.plot(t_s, np.degrees(tr['angle']), color='C2', alpha=0.5, lw=0.7)
    ax.axvline(delay_start, color='gray', ls='--', lw=0.6)
    ax.axvline(delay_end, color='gray', ls='--', lw=0.6)
    ax.axhline(np.degrees(cue), color='k', ls=':', alpha=0.5)
    ax.set_xlabel('time (s)')
    ax.set_ylabel('decoded angle (deg)')
    ax.set_title(f'Decoded angle, n={len(trials)} trials')
    ax.set_xlim(delay_start - 0.5, delay_end + 0.2)
    ax.set_ylim(-90, 90)

    # Variance vs time within the delay
    ax = axes[1]
    p_dt = PARAMS['dt']
    i_start = int(trials[0]['events']['stimoff'] / p_dt)
    i_end = int(trials[0]['events']['delayend'] / p_dt)
    angles = np.array([tr['angle'][i_start:i_end] for tr in trials])  # (trials, T)
    # Wrap to (-pi, pi) relative to cue, then variance in degrees^2
    err = np.angle(np.exp(1j * (angles - cue)))
    var_deg2 = np.degrees(err).var(axis=0)
    t_delay = (np.arange(len(var_deg2)) * p_dt) / 1000.0
    ax.plot(t_delay, var_deg2, color='C2')
    ax.set_xlabel('time within delay (s)')
    ax.set_ylabel(r'variance of decoded angle (deg$^2$)')
    ax.set_title('Diffusion of the bump')

    plt.tight_layout()
    if save_name:
        for ext in ('.png', '.svg'):
            plt.savefig(os.path.join(SAVE_DIR, save_name + ext), dpi=150, bbox_inches='tight')
    plt.show()


plot_drift(trials, cue, save_name='wimmer_drift_demo')

The right panel is expected to grow approximately **linearly** with time within the delay — the signature of a 1D diffusion process. With $n=10$ trials the curve typically bounces around because sample variance is noisy when the sample is small; do not over-interpret bumps or dips. Crank `n_trials` up to a few hundred (each trial takes a couple of seconds) and the curve becomes a clean straight line, matching Wimmer Figure 5B and Compte 2000 Figure 5B.

There is also a small early-delay transient visible in some runs: as the network relaxes from the cue-driven state (wider, taller bump during the cue) to the natural attractor (narrower, lower bump during the delay), the population-vector readout can wobble briefly. This is not diffusion *per se*. It disappears within the first few hundred milliseconds of the delay.

## Sanity check — the bump is a continuous attractor

If the model is a true *continuous* bump attractor, presenting the cue at any angle should produce a bump centred at that angle (up to noise drift). Run one trial per cue location, and compare the late-delay decoded angle to the cue.

In [ ]:
cues = np.linspace(-np.pi, np.pi, 8, endpoint=False)
trials_by_cue = [run_trial(PARAMS, cue_angle=c, seed=200 + k)
                 for k, c in enumerate(cues)]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6),
                          gridspec_kw=dict(width_ratios=[2, 1]))

ax = axes[0]
theta_deg = np.degrees(trials_by_cue[0]['theta'])
p_dt = PARAMS['dt']
i_late = int((PARAMS['delayend'] - 100) / p_dt)
for tr, c in zip(trials_by_cue, cues):
    ax.plot(theta_deg, tr['rE'][i_late], label=f'{np.degrees(c):+.0f}°')
ax.set_xlabel('preferred angle (deg)')
ax.set_ylabel('E rate at late delay (a.u.)')
ax.set_title('Late-delay activity profile, one trial per cue')
ax.set_xlim(-180, 180)
ax.legend(loc='upper right', fontsize=8, ncol=2)

ax = axes[1]
decoded_late = np.array([np.degrees(tr['angle'][i_late]) for tr in trials_by_cue])
ax.plot(np.degrees(cues), decoded_late, 'o', color='C2')
ax.plot([-180, 180], [-180, 180], '--', color='k', alpha=0.4, label='identity')
ax.set_xlabel('cue angle (deg)')
ax.set_ylabel('decoded angle, late delay (deg)')
ax.set_title('Continuous representation')
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.legend()

plt.tight_layout()
for ext in ('.png', '.svg'):
    plt.savefig(os.path.join(SAVE_DIR, 'wimmer_continuous_attractor' + ext),
                dpi=150, bbox_inches='tight')
plt.show()

If the model works correctly, the right panel falls almost exactly on the identity line (with a small noise-induced scatter), and the left panel shows eight bumps centred on the eight cue locations. This is what makes the model a *continuous* attractor: the location of the bump is a free parameter set by the cue.

## What this notebook does and does not include

**Done** — a faithful Python port of Wimmer et al. (2014) Supplementary Code 1. Same parameters, same equations, same noise convention, same trial protocol. The qualitative reproductions cover: bump formation, bump persistence through the delay, bump diffusion across trials, attractor continuity across cues.

**Not done** (these were not in the released code, only in the published paper analyses):
- The full surrogate dataset of 16 000 trials used to reproduce the experimental tuning-bias and rate–behaviour analyses.
- Single-neuron tuning curves and noise-correlation analyses.
- Comparison with the discrete-attractor and decaying-bump alternatives (Supplementary Codes 2 and 3).

These are straightforward extensions on top of `run_trial`. The simplest way to get the variance-versus-delay figure is to bump `n_trials` in the diffusion section to a few hundred.